In [16]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import math

In [33]:
# Lista das estruturas a serem lidas
STRUCTURE_FILES = {
    'AVL': 'AVL_metrics.csv',
    'RedBlack': 'RedBlack_metrics.csv',
    'ChainedHash': 'ChainedHash_metrics.csv',
    'OpenHash': 'OpenHash_metrics.csv'
}

# Lista das métricas que queremos plotar
METRICS_TO_PLOT = [
    'tempo_ms',
    'comparacoes',
    'insercoes',
    'rotacoes',
    'altura',
    'colisoes',
    'rehashes',
    'fator_carga',
    # 'palavras_totais',
    'palavras_unicas'
]
# Cores fixas por estrutura
STRUCTURE_COLORS = {
    'AVL': '#1f77b4',
    'RedBlack': '#ff7f0e',
    'ChainedHash': '#2ca02c',
    'OpenHash': '#d62728'
}

# Label personalizado com título e unidade para cada métrica
METRIC_LABELS = {
    'tempo_ms':        {'title': 'Tempo',           'unidade': 'ms'},
    'comparacoes':     {'title': 'Comparações',     'unidade': 'unidade'},
    'insercoes':       {'title': 'Inserções',       'unidade': 'unidade'},
    'rotacoes':        {'title': 'Rotações',        'unidade': 'unidade'},
    'altura':          {'title': 'Altura',          'unidade': 'unidade'},
    'colisoes':        {'title': 'Colisões',        'unidade': 'unidade'},
    'rehashes':        {'title': 'Rehashes',        'unidade': 'unidade'},
    'fator_carga':     {'title': 'Fator de Carga',  'unidade': ''},
    'palavras_totais': {'title': 'Palavras Totais', 'unidade': 'unidade'},
    'palavras_unicas': {'title': 'Palavras Únicas', 'unidade': 'unidade'}
}

In [44]:
def read_metrics_from_folder(folder_path):
    """Reads all CSVs from the folder and returns a dictionary of DataFrames."""
    data = {}
    for name, filename in STRUCTURE_FILES.items():
        file_path = os.path.join(folder_path, filename)
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            data[name] = df
        else:
            print(f"Warning: File not found: {file_path}")
    return data

def clean_metrics(df):
    """Substitui valores -1 por NaN para evitar plotagem."""
    df_clean = df.copy()
    for metric in METRICS_TO_PLOT + ['palavras_totais']:
        if metric in df_clean.columns:
            df_clean[metric] = df_clean[metric].replace(-1, pd.NA)
    return df_clean

def plot_comparison_graphs(data_dict, output_folder=None):
    """
    Gera uma imagem com subplots para cada metrica por arquivo analisado.
    """
    # Pegamos os nomes dos arquivos de texto como referência
    sample_df = next(iter(data_dict.values()))
    file_names = sample_df['arquivo']

    for idx, arquivo in enumerate(file_names):
        num_metrics = len(METRICS_TO_PLOT)
        cols = 3
        rows = math.ceil(num_metrics / cols)

        fig, axs = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
        axs = axs.flatten()

        for i, metric in enumerate(METRICS_TO_PLOT):
            estruturas = []
            valores = []

            for estrutura, df in data_dict.items():
                df_clean = clean_metrics(df)
                valor = df_clean[metric].iloc[idx]
                if pd.notna(valor):
                    estruturas.append(estrutura)
                    valores.append(valor)

            ax = axs[i]
            colors = [STRUCTURE_COLORS.get(estr, 'gray') for estr in estruturas]
            bars = ax.bar(estruturas, valores, color=colors)

            # Adiciona os valores acima das barras
            for bar, valor in zip(bars, valores):
                altura = bar.get_height()
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    altura,
                    f'{valor:.0f}',
                    ha='center',
                    va='bottom',
                    fontsize=8
                )

            label_info = METRIC_LABELS.get(metric, {'title': metric, 'unidade': ''})
            ax.set_title(label_info['title'], fontweight='bold')
            ax.set_ylabel(label_info['unidade'])

        # Esconde subplots vazios se houver
        for j in range(i + 1, len(axs)):
            fig.delaxes(axs[j])

        # Título principal
        fig.suptitle(f'Métricas para o livro "{arquivo}"', fontsize=16, fontweight='bold')

        # Texto com total de palavras
        palavras_totais = sample_df['palavras_totais'].iloc[idx]
        fig.text(
            0.5,
            0.01,
            f'Total de palavras no livro "{arquivo}": {int(palavras_totais):,}'.replace(',', '.'),
            ha='center',
            fontsize=10,
            style='italic',
            fontweight ='bold'
        )

        fig.tight_layout(rect=[0, 0.05, 1, 0.95])  # Ajusta espaço para título e texto inferior

        # Salvar ou mostrar
        if output_folder:
            os.makedirs(output_folder, exist_ok=True)
            filename_safe = arquivo.replace('/', '_').replace('\\', '_')
            plt.savefig(os.path.join(output_folder, f'{filename_safe}.png'))
            plt.close(fig)
        else:
            plt.show()


def run_analysis(folder_path, output_folder=None):
    """
    Main function to run the analysis.
    """
    data = read_metrics_from_folder(folder_path)
    if not data:
        print("No data found.")
        return
    plot_comparison_graphs(data, output_folder)


In [47]:
# Exemplo de uso
if __name__ == "__main__":
    folders = [
        'biblia',
        'dom-casmurro',
        'manifesto',
        'riqueza',
        'secret-garden',
        'sherlock'
    ]
    
    for folder in folders:
        run_analysis(folder, "graficos")
    